# 04 · Tuning de los tres mejores candidatos

**Qué se ajusta.** Las tres familias con mejor PR-AUC por evento en la CV del notebook 03 (`docs/candidatos_top3.json`),
con **30 configuraciones cada una**. Si una de las tres no se puede ajustar en un tiempo razonable (TabPFN: una sola
CV completa toma ~7 h), se documenta y pasa la siguiente del ranking.

**Método: Optuna con TPE y poda por mediana.**

- **TPE** (*Tree-structured Parzen Estimator*) modela qué regiones del espacio dieron buenas configuraciones y propone
  las siguientes ahí. Con un costo de minutos por configuración, encuentra buenas zonas con muchas menos pruebas que
  una búsqueda en grilla o aleatoria. Las primeras 8 configuraciones son aleatorias, para explorar.
- **`MedianPruner`**: después de cada pliegue se reporta la PR-AUC por evento acumulada y se corta la configuración si
  va por debajo de la mediana de las anteriores en ese pliegue. Los pliegues son expansivos, así que el primero es el
  más barato: las configuraciones malas se descartan sin pagar la CV completa.
- **Escalas:** logarítmica para tasas de aprendizaje, regularización, pesos y número de árboles; lineal para
  profundidades, fracciones y épocas.

**Presupuesto.** Durante la búsqueda cada configuración entrena con todos los discos que fallan y el **5 %** de los
sanos (en el notebook 03 era el 15 %); la validación sigue siendo la misma CV con **todas** sus filas. Con el 15 %,
30 configuraciones de XGBoost tomarían ~6,5 h por familia; con el 5 %, alrededor de un tercio. La **mejor
configuración se reentrena con el protocolo del notebook 03** (15 %) para compararla de forma justa con la versión
por defecto. La mejora solo se acepta si supera la desviación estándar entre pliegues.

**Registro.** Cada familia es una corrida padre `tuning_<familia>` en MLflow y cada configuración es una corrida hija
anidada, con sus hiperparámetros, semilla, datos, métricas por pliegue y modelo (las podadas quedan registradas sin
modelo). Los estudios viven en `/datos/optuna/pset3.db`: los scripts `scripts/tunear.py` pueden correr familias en otras
GPUs y este notebook retoma lo que dejen.

In [ ]:
import os, sys, json, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import optuna

sys.path.insert(0, str(Path.cwd().parent))
from src import datos, pipeline as pl, experimentos as exp, tuning

optuna.logging.set_verbosity(optuna.logging.WARNING)
pd.set_option("display.width", 220, "display.max_columns", 40, "display.float_format", "{:,.4f}".format)
sns.set_theme(style="whitegrid", context="notebook")

DATOS = Path(os.environ.get("DATOS_DIR", "/datos"))
FIG = Path("../docs/figuras"); FIG.mkdir(parents=True, exist_ok=True)
SEMILLA, FRAC_TUNING, FRAC_FINAL = pl.SEMILLA, 0.05, 0.15
STORAGE = tuning.storage_por_defecto(DATOS)
T0 = time.time()

mlflow = pl.configurar_mlflow()
print("MLflow:", mlflow.get_tracking_uri(), "· experimento:", pl.NOMBRE_EXPERIMENTO, "· commit:", pl.commit_git())
print("Estudios de Optuna:", STORAGE)

## 1. Qué familias se ajustan

In [ ]:
candidatos_03 = pd.DataFrame(exp.resultados(DATOS / "oof" / "03").values()).set_index("nombre")
ranking = (candidatos_03[(candidatos_03["fase"] == "candidato") & ~candidatos_03.index.str.contains("sin_degradacion")]
           .sort_values("cv_pr_auc_evento_media", ascending=False))
top3 = json.loads(Path("../docs/candidatos_top3.json").read_text())["familias"]

FAMILIAS = []
for familia in ranking["familia"]:
    if len(FAMILIAS) == 3:
        break
    if familia in tuning.ESPACIOS:
        FAMILIAS.append(familia)
    else:
        print(f"{familia}: entre las mejores, pero no se ajusta (una CV completa toma horas); pasa la siguiente")
print("Top 3 del notebook 03:", top3, "→ se ajustan:", FAMILIAS)
ranking[["familia", "cv_pr_auc_evento_media", "cv_pr_auc_evento_std", "brecha_train_cv"]]

## 2. Búsqueda (30 configuraciones por familia)

In [ ]:
CTX_TUNING = exp.preparar(DATOS, DATOS / "oof" / "04", frac_sanos=FRAC_TUNING)
ESTUDIOS = {}
for familia in FAMILIAS:
    t = time.time()
    ESTUDIOS[familia] = tuning.tunear(CTX_TUNING, familia, STORAGE, n_trials=tuning.N_TRIALS)
    st = ESTUDIOS[familia]
    estados = pd.Series([t_.state.name for t_ in st.trials]).value_counts().to_dict()
    print(f"{familia}: {len(st.trials)} configuraciones {estados} · mejor CV {st.best_value:.4f} "
          f"(trial {st.best_trial.number}) · {(time.time()-t)/60:.1f} min en esta ejecución")

In [ ]:
fig, axes = plt.subplots(1, len(FAMILIAS), figsize=(5 * len(FAMILIAS), 3.8), squeeze=False)
for ax, familia in zip(axes[0], FAMILIAS):
    df = tuning.resumen_estudio(ESTUDIOS[familia]).sort_values("number")
    completos = df[df["state"] == "COMPLETE"]
    ax.scatter(completos["number"], completos["value"], s=18, label="configuración completa")
    podados = df[df["state"] == "PRUNED"]
    ax.scatter(podados["number"], np.full(len(podados), completos["value"].min()), marker="x", color="grey",
               s=18, label="podada")
    ax.plot(completos["number"], completos["value"].cummax(), color="k", lw=1.2, label="mejor hasta ahí")
    ax.set_title(familia); ax.set_xlabel("configuración"); ax.set_ylabel("PR-AUC por evento (CV)")
axes[0][0].legend(fontsize=8)
fig.suptitle("Historia de la búsqueda (TPE + poda por mediana)")
fig.tight_layout(); fig.savefig(FIG / "04_historia_tuning.png", dpi=150); plt.show()

In [ ]:
for familia in FAMILIAS:
    st = ESTUDIOS[familia]
    try:
        imp = optuna.importance.get_param_importances(st)
        print(f"{familia} · importancia de hiperparámetros (fANOVA):",
              ", ".join(f"{k} {v:.2f}" for k, v in imp.items()))
    except Exception as e:  # noqa: BLE001 - con pocas configuraciones completas fANOVA puede fallar
        print(f"{familia}: importancia no disponible ({type(e).__name__})")
    cols = ["number", "value", "state"] + [c for c in tuning.resumen_estudio(st).columns if c.startswith("params_")]
    display(tuning.resumen_estudio(st)[cols].head(5))

## 3. La mejor configuración, con el protocolo del notebook 03

La búsqueda usó el 5 % de los discos sanos para entrenar. Para comparar con la versión por defecto de forma justa, la
mejor configuración de cada familia se reentrena con **exactamente el mismo protocolo del notebook 03** (15 % de sanos,
misma CV, misma semilla) y se registra como `<familia>_optimizado` (`fase=tuning`).

In [ ]:
CTX_FINAL = exp.preparar(DATOS, DATOS / "oof" / "04", frac_sanos=FRAC_FINAL)
OPTIMIZADOS = {}
for familia in FAMILIAS:
    st = ESTUDIOS[familia]
    params = tuning.ESPACIOS[familia](optuna.trial.FixedTrial(st.best_params))  # hiperparámetros del modelo
    OPTIMIZADOS[familia] = exp.correr(
        CTX_FINAL, f"{familia}_optimizado", "tuning", familia, params=params,
        tags={"rol": "mejor_configuracion", "trial_origen": str(st.best_trial.number)},
        params_registro={f"hp_{k}": v for k, v in st.best_params.items()})

In [ ]:
filas = []
for familia in FAMILIAS:
    base, opt = candidatos_03.loc[familia], OPTIMIZADOS[familia]
    delta = opt["cv_pr_auc_evento_media"] - base["cv_pr_auc_evento_media"]
    filas.append({"familia": familia,
                  "cv_por_defecto": base["cv_pr_auc_evento_media"], "std_por_defecto": base["cv_pr_auc_evento_std"],
                  "cv_optimizado": opt["cv_pr_auc_evento_media"], "std_optimizado": opt["cv_pr_auc_evento_std"],
                  "mejora": delta, "mejora_relativa": delta / base["cv_pr_auc_evento_media"],
                  "mejora_supera_std": delta > base["cv_pr_auc_evento_std"],
                  "train_optimizado": opt["train_pr_auc_evento"], "brecha_optimizado": opt["brecha_train_cv"]})
comparacion = pd.DataFrame(filas).set_index("familia")
Path("../docs/tuning_resultado.json").write_text(json.dumps({
    familia: {"mejores_hiperparametros": ESTUDIOS[familia].best_params, "trial": ESTUDIOS[familia].best_trial.number,
              "run_id_optimizado": OPTIMIZADOS[familia]["run_id"],
              **{k: (bool(v) if isinstance(v, (bool, np.bool_)) else float(v))
                 for k, v in comparacion.loc[familia].items()}}
    for familia in FAMILIAS}, indent=2, ensure_ascii=False))
print(f"Tiempo total del notebook: {(time.time()-T0)/60:.1f} min")
comparacion